# Forecasting U.S. House Elections with Historical Priors

**COSC 274: Machine Learning & Statistical Data Analysis · Final Project**

**Research question.** Does using older elections as a *prior* produce better-calibrated forecasts than simply pooling all the data together?

For every U.S. House race, this notebook predicts two things using only information available before Election Day:

| Task | Target | Models |
|---|---|---|
| Classification | Does the Democratic candidate win? | Logistic regression, random forest, Bayesian classifier with a historical prior |
| Regression | Two-party vote margin | Least squares, least absolute deviations (LAD), Bayesian linear regression with a historical prior |

**Data.** MIT Election Data and Science Lab (MEDSL), *U.S. House 1976–2024*, Harvard Dataverse.

**How this notebook fits into the project.**
- `src/build_dataset.py` turns the raw MEDSL returns (one row per candidate) into `data/processed/house_races.csv` (one row per district per election) and computes the features.
- `notebooks/01_explore.ipynb` is the exploratory analysis. Its findings are cited below wherever they motivate a modeling decision.
- This notebook is the main workflow: features, splits, models, evaluation, and conclusions.

**Contents**
1. Setup
2. Data
3. Features, splits, and the baseline
4. Classification: who wins?
5. Regression: by how much?
6. Research question: historical prior vs. pooled data
7. Final evaluation on the test and stress-test elections
8. Discussion and conclusion

## 1. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression, QuantileRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, brier_score_loss, f1_score, log_loss,
                             mean_absolute_error, mean_squared_error, precision_score,
                             recall_score, roc_auc_score)

SEED = 274

ROOT = Path("..")                                   # notebook lives in notebooks/
DATA = ROOT / "data" / "processed" / "house_races.csv"
RESULTS = ROOT / "results"
FIGS = RESULTS / "figures"
FIGS.mkdir(parents=True, exist_ok=True)

# Experimental design (see README)
PRIOR_YEARS  = list(range(1978, 1999, 2))           # Bayesian models: these elections set the prior
UPDATE_YEARS = list(range(2000, 2015, 2))           # Bayesian models: these elections update it
TRAIN_YEARS  = PRIOR_YEARS + UPDATE_YEARS           # every model sees the same 1978–2014 elections
VAL_YEAR, TEST_YEAR, STRESS_YEAR = 2016, 2018, 2022

plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.float_format", "{:.3f}".format)

## 2. Data

`src/build_dataset.py` already did the heavy lifting on the raw MEDSL file:

- repaired the quote-wrapped Dataverse export and parsed all 33,805 candidate rows;
- kept, for each seat, the election that actually decided it (regular general, special general, or runoff, which matters for Texas in 1996 and 2006);
- merged New York fusion-ticket lines, dropped ballot artifacts ("BLANK VOTE", "EXHAUSTED BALLOT"), and excluded the D.C. delegate;
- computed the features below, all **signed from the Democratic point of view** (positive favors Democrats, negative favors Republicans), so the models treat the two parties symmetrically.

| Column | Meaning |
|---|---|
| `margin` | two-party margin, (D − R) / (D + R); ±1 when only one major party ran |
| `dem_win` | 1 if the Democrat won (classification target) |
| `contested` | both major parties had a candidate |
| `margin_lag1`, `margin_lag2` | margin in the same district one and two cycles earlier |
| `contested_lag1` | whether the previous race was contested |
| `incumbent` | +1 Democratic incumbent running, −1 Republican incumbent, 0 open seat |
| `midterm`, `pres_party` | midterm year; party of the sitting president (+1 D, −1 R) |
| `midterm_x_pres` | midterm × president's party: the midterm-penalty term |
| `redistrict_year` | first election under new post-census district lines |

In [ ]:
if not DATA.exists():
    raise FileNotFoundError(f"{DATA} not found. From the repo root, run: "
                            "python src/build_dataset.py --input data/raw/<medsl file>")

races = pd.read_csv(DATA)
per_year = races.groupby("year").size()
assert (per_year == 435).all(), "Expected 435 races in every cycle; rerun src/build_dataset.py"

print(f"{len(races):,} races across {races['year'].nunique()} cycles "
      f"({races['year'].min()}–{races['year'].max()}), 435 per cycle")
races.head()

## 3. Features, splits, and the baseline

### 3.1 The modeling table

Three preparation steps:

1. **Drop races with no previous result in the district.** That is every 1976 race (there is no 1974 data) plus districts newly created after reapportionment. Without `margin_lag1` there is nothing to forecast from.
2. **Fill missing two-cycle lags.** When `margin_lag2` is unavailable it is set equal to `margin_lag1`, and a `has_lag2` flag records that it was filled.
3. **Flag competitive races for evaluation.** A race is *competitive* if it was contested and decided by fewer than 10 points. This uses the actual result, so it is only ever used to slice the evaluation, never as a feature.

Races won by a third-party or independent candidate (12 in total, 8 of them Bernie Sanders's Vermont seat) are excluded, since neither class describes them.

In [ ]:
FEATURES = ["margin_lag1", "margin_lag2", "has_lag2", "contested_lag1",
            "incumbent", "midterm", "pres_party", "midterm_x_pres", "redistrict_year"]

def as_int_flag(s):
    # CSV booleans can load as bools or strings depending on missing values; normalise to 0/1.
    return s.astype(str).str.upper().eq("TRUE").astype(int)

df = races.copy()
df["contested"] = as_int_flag(df["contested"]).astype(bool)

no_lag = df["margin_lag1"].isna()
third_party = df["winner_party"] == "O"
df = df[~no_lag & ~third_party].copy()

df["has_lag2"] = df["margin_lag2"].notna().astype(int)
df["margin_lag2"] = df["margin_lag2"].fillna(df["margin_lag1"])
df["contested_lag1"] = as_int_flag(df["contested_lag1"])
df["competitive"] = df["contested"] & (df["margin"].abs() < 0.10)

print(f"Dropped {no_lag.sum():,} races with no previous result and "
      f"{(third_party & ~no_lag).sum()} third-party wins; {len(df):,} races remain.")
df[FEATURES].describe().T[["mean", "std", "min", "max"]]

### 3.2 Splits

The splits are by **time**, never random, because the real task is forecasting a future election from past ones. Hyperparameters are chosen on 2016 only. The 2018 test election and the 2022 stress test are not touched until Section 7.

2022 is a deliberate stress test: it was the first election under post-2020-census maps, so most districts' `margin_lag1` describes a differently drawn district.

In [ ]:
def years(data, yrs):
    yrs = yrs if isinstance(yrs, list) else [yrs]
    return data[data["year"].isin(yrs)]

split_years = {
    "prior (1978–1998)":   PRIOR_YEARS,
    "update (2000–2014)":  UPDATE_YEARS,
    "train (1978–2014)":   TRAIN_YEARS,
    "validation (2016)":   VAL_YEAR,
    "test (2018)":         TEST_YEAR,
    "stress test (2022)":  STRESS_YEAR,
}
pd.DataFrame({
    name: {"races": len(s := years(df, y)),
           "Democratic win rate": s["dem_win"].mean(),
           "contested share": s["contested"].mean(),
           "competitive share": s["competitive"].mean()}
    for name, y in split_years.items()
}).T

### 3.3 Evaluation metrics and the baseline

Because most districts are safe, plain accuracy flatters every model (Lecture 7). Each classifier is therefore scored on:

- **log loss** (cross-entropy) and **Brier score**: how good the predicted probabilities are, not just the labels;
- **ROC AUC**: how well the model ranks Democratic wins above Republican wins;
- **calibration error (ECE)**: the average gap between predicted probability and observed win rate, across 10 probability bins. This is the measure most directly tied to the research question;
- **accuracy on all races** and **accuracy on competitive races**, where the real difficulty is;
- **macro F1** (the average of the F1 scores for each party, so neither party counts as "the positive class"), plus precision and recall for Democratic wins.

The **baseline** predicts that whichever party won the district last time wins again. It outputs labels, not probabilities, so its probability-based metrics are left blank.

In [ ]:
def expected_calibration_error(y, p, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, bins - 1)
    return sum(abs(y[idx == b].mean() - p[idx == b].mean()) * (idx == b).mean()
               for b in range(bins) if (idx == b).any())

def clf_metrics(data, p=None, yhat=None):
    # p: predicted P(Democrat wins). Pass yhat instead for models that only output labels.
    y = data["dem_win"].to_numpy()
    comp = data["competitive"].to_numpy()
    yhat = (p >= 0.5).astype(int) if yhat is None else np.asarray(yhat).astype(int)
    m = {
        "log loss":               log_loss(y, p) if p is not None else np.nan,
        "Brier score":            brier_score_loss(y, p) if p is not None else np.nan,
        "ROC AUC":                roc_auc_score(y, p) if p is not None else np.nan,
        "calibration error":      expected_calibration_error(y, p) if p is not None else np.nan,
        "accuracy":               accuracy_score(y, yhat),
        "accuracy (competitive)": accuracy_score(y[comp], yhat[comp]),
        "macro F1":               f1_score(y, yhat, average="macro"),
        "precision (D win)":      precision_score(y, yhat),
        "recall (D win)":         recall_score(y, yhat),
    }
    return m

def baseline_labels(data):
    return (data["margin_lag1"] > 0).astype(int).to_numpy()

train, val = years(df, TRAIN_YEARS), years(df, VAL_YEAR)

pd.DataFrame({
    "baseline, train 1978–2014": clf_metrics(train, yhat=baseline_labels(train)),
    "baseline, validation 2016": clf_metrics(val, yhat=baseline_labels(val)),
}).T

The baseline is strong: it calls about 91% of 1978–2014 races and 97% of 2016 races correctly. 2016 was a status-quo year in which few seats changed hands, which is exactly when "same party wins again" works best. Its weakness is competitive races, where it is right only about 62% of the time in training. Those are the races where a model has to add something.

One caution for everything that follows: 2016 has only 32 competitive races, so each one moves competitive accuracy by about 3 points. Differences of a race or two on that slice are noise, which is one more reason to tune on log loss rather than accuracy.

## 4. Classification: who wins?

All classifiers train on the same 1978–2014 elections and are tuned on 2016 by **validation log loss**, since well-calibrated probabilities are the goal.

In [ ]:
def Xy(data, target):
    return data[FEATURES].to_numpy(dtype=float), data[target].to_numpy()

X_tr, y_tr = Xy(train, "dem_win")
X_val, y_val = Xy(val, "dem_win")
print(f"train: {X_tr.shape[0]:,} races × {X_tr.shape[1]} features   validation: {X_val.shape[0]:,} races")

### 4.1 Logistic regression (cross-entropy loss)

Logistic regression models P(Democrat wins) = σ(x̃ᵀw) and is fit by minimizing cross-entropy (Lecture 6). scikit-learn adds an L2 penalty whose strength is set by `C`, where **C = 1/λ**: small `C` means strong regularization (Lecture 5). Features are standardized first, so the penalty treats them equally and the coefficients are comparable.

In [ ]:
Cs = np.logspace(-3, 3, 13)
rows = []
for C in Cs:
    model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))
    model.fit(X_tr, y_tr)
    rows.append({"C": C,
                 "train log loss": log_loss(y_tr, model.predict_proba(X_tr)[:, 1]),
                 "validation log loss": log_loss(y_val, model.predict_proba(X_val)[:, 1])})
lr_tuning = pd.DataFrame(rows)
# Validation loss flattens out once the penalty is weak, so pick the strongest regularization
# (smallest C) whose validation loss is within 0.001 of the best: simpler model, same performance.
tol = lr_tuning["validation log loss"].min() + 0.001
best_C = lr_tuning.loc[lr_tuning["validation log loss"] <= tol, "C"].min()

fig, ax = plt.subplots()
ax.semilogx(lr_tuning["C"], lr_tuning["train log loss"], marker="o", label="train (1978–2014)")
ax.semilogx(lr_tuning["C"], lr_tuning["validation log loss"], marker="o", label="validation (2016)")
ax.axvline(best_C, color="gray", linestyle=":", label=f"selected C = {best_C:.3g}")
ax.set(title="Logistic regression: choosing the regularization strength",
       xlabel="C (larger = weaker L2 penalty)", ylabel="log loss")
ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(FIGS / "lr_tuning.png", dpi=150)
plt.show()

lr = make_pipeline(StandardScaler(), LogisticRegression(C=best_C, max_iter=5000)).fit(X_tr, y_tr)
print(f"Selected C = {best_C:.3g}")

**What the model learned.** Because the features are standardized, each coefficient is the change in the log-odds of a Democratic win for a one-standard-deviation increase in that feature.

In [ ]:
lr_coefs = pd.Series(lr[-1].coef_[0], index=FEATURES).sort_values()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(lr_coefs.index, lr_coefs.values, color=["indianred" if v < 0 else "steelblue" for v in lr_coefs])
ax.axvline(0, color="black", linewidth=0.8)
ax.set(title="Logistic regression coefficients (standardized features)",
       xlabel="change in log-odds of a Democratic win per 1 SD")
fig.tight_layout(); fig.savefig(FIGS / "lr_coefficients.png", dpi=150)
plt.show()

lr_coefs.to_frame("coefficient").iloc[::-1]

Three things stand out:

- **Incumbency is the strongest predictor**, followed by the previous margin. Together they carry most of the signal, consistent with the exploratory analysis.
- **The midterm penalty is clearly negative**, as expected. `midterm_x_pres` is +1 in a midterm under a Democratic president, so a negative coefficient means Democrats do worse in those years and better in midterms under a Republican president. This matches the swing found in `01_explore.ipynb`, Section 5.
- **The redistricting flag adds almost nothing on its own.** Its real effect should be to make the model trust `margin_lag1` less in redistricting years, which is an interaction a linear model can't express without being given it explicitly. That is one motivation for the random forest.

The selected regularization is moderate: validation log loss is flat for any C above about 0.3, because with only nine features and over 8,000 races the model has little room to overfit. Validation log loss is also *lower* than training log loss, which simply reflects that 2016 was an easier year to predict than the 1978–2014 average.

### 4.2 Random forest

A random forest averages many decision trees, each grown on a bootstrap sample of the races with a random subset of features considered at each split. Unlike logistic regression it can learn **interactions** without being told about them, for example that incumbency matters more in close districts than in safe ones. Trees don't need standardized features.

Two hyperparameters control how complex each tree can get: the maximum depth, and the minimum number of races allowed in a leaf. Both are tuned on 2016.

In [ ]:
rf_rows = []
for depth in [3, 5, 8, 12, None]:
    for leaf in [1, 10, 50]:
        model = RandomForestClassifier(n_estimators=300, max_depth=depth, min_samples_leaf=leaf,
                                       random_state=SEED, n_jobs=-1).fit(X_tr, y_tr)
        rf_rows.append({"max_depth": "none" if depth is None else depth, "min_samples_leaf": leaf,
                        "train log loss": log_loss(y_tr, model.predict_proba(X_tr)[:, 1]),
                        "validation log loss": log_loss(y_val, model.predict_proba(X_val)[:, 1])})
rf_tuning = pd.DataFrame(rf_rows)
best = rf_tuning.loc[rf_tuning["validation log loss"].idxmin()]
best_depth = None if best["max_depth"] == "none" else int(best["max_depth"])
best_leaf = int(best["min_samples_leaf"])

rf = RandomForestClassifier(n_estimators=500, max_depth=best_depth, min_samples_leaf=best_leaf,
                            random_state=SEED, n_jobs=-1).fit(X_tr, y_tr)
print(f"Selected max_depth = {best['max_depth']}, min_samples_leaf = {best_leaf}")
rf_tuning.pivot(index="max_depth", columns="min_samples_leaf", values="validation log loss")

The tuning grid shows the usual trade-off between underfitting and overfitting. Shallow trees (depth 3) underfit. Fully grown trees with single-race leaves overfit badly: validation log loss more than doubles, because pure leaves output probabilities of exactly 0 or 1, and cross-entropy punishes a confident wrong prediction very heavily. Moderate depth works best, and at that depth the leaf size barely matters.

**Which features does the forest rely on?** Permutation importance shuffles one feature at a time in the 2016 validation data and measures how much the log loss gets worse. Unlike the forest's built-in importances, it is measured on held-out data.

In [ ]:
perm = permutation_importance(rf, X_val, y_val, scoring="neg_log_loss", n_repeats=20, random_state=SEED)
rf_importance = pd.Series(perm.importances_mean, index=FEATURES).sort_values()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(rf_importance.index, rf_importance.values, xerr=perm.importances_std[rf_importance.index.map(FEATURES.index)],
        color="seagreen")
ax.set(title="Random forest: permutation importance on 2016",
       xlabel="increase in validation log loss when the feature is shuffled")
fig.tight_layout(); fig.savefig(FIGS / "rf_importance.png", dpi=150)
plt.show()

**Important caveat when reading this chart.** The national features (`midterm`, `pres_party`, `midterm_x_pres`, `redistrict_year`) take the same value for every race in a single election, so shuffling them within 2016 changes nothing and their importance is zero *by construction*. This chart can only rank the district-level features: the previous margin and incumbency dominate, the two-cycle lag adds a little, and the rest add almost nothing. The national features' effects are better judged from the logistic regression coefficients in Section 4.1.

### 4.3 Bayesian classifier with a historical prior

*To be completed after the Bayesian modeling lectures, following the formulation used in class.*

Plan: elections from 1978–1998 set the prior over the model's parameters, and 2000–2014 update it into a posterior. Predictions for 2016 (validation) come from the posterior, and the strength of the prior is tuned on 2016 like any other hyperparameter.

### 4.4 Validation comparison so far

The table compares the models on 2016. The reliability diagram (calibration plot) groups races by predicted probability and plots the share Democrats actually won: a perfectly calibrated model lies on the diagonal.

In [ ]:
val_preds = {"logistic regression": lr.predict_proba(X_val)[:, 1],
             "random forest":       rf.predict_proba(X_val)[:, 1]}

clf_val = pd.DataFrame({"baseline (same party wins)": clf_metrics(val, yhat=baseline_labels(val)),
                        **{name: clf_metrics(val, p) for name, p in val_preds.items()}}).T
clf_val.to_csv(RESULTS / "validation_classification.csv")
print(f"2016 has {val['competitive'].sum()} competitive races, so one race moves competitive accuracy by "
      f"{1 / val['competitive'].sum():.1%}.")

def reliability(ax, y, p, label, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, bins - 1)
    pts = [(p[idx == b].mean(), y[idx == b].mean(), (idx == b).sum()) for b in range(bins) if (idx == b).any()]
    xs, ys, ns = map(np.array, zip(*pts))
    line, = ax.plot(xs, ys, label=label)
    ax.scatter(xs, ys, s=12 * np.sqrt(ns), color=line.get_color(), alpha=0.8, zorder=3)
    return ns

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.plot([0, 1], [0, 1], color="gray", linestyle="--", linewidth=1, label="perfect calibration")
for name, p in val_preds.items():
    reliability(ax, y_val, p, name)
ax.set(title="Calibration on 2016 (validation)\nmarker area ∝ number of races in the bin", xlabel="predicted P(Democrat wins)",
       ylabel="share of races Democrats won", xlim=(0, 1), ylim=(0, 1))
ax.legend(frameon=False, loc="upper left")
fig.tight_layout(); fig.savefig(FIGS / "calibration_validation.png", dpi=150)
plt.show()

clf_val

**Accuracy:** neither model beats the baseline on 2016. In a status-quo year, "same party wins again" is very hard to beat on labels alone, and on 32 competitive races the gaps amount to one or two races.

**Probabilities:** this is where the models add value. Both rank races almost perfectly (ROC AUC around 0.99) and give useful probabilities rather than bare labels. The random forest is slightly better than logistic regression on log loss, Brier score, and calibration error, suggesting it picks up some interaction the linear model misses.

**Calibration:** the reliability curves are steeper than the diagonal. Races given a 60–70% chance went to Democrats almost every time, and races given 20–30% rarely did, so **both models are underconfident on 2016.** This connects directly to the exploratory analysis: results have become much more predictable since the 1980s, and models trained on all of 1978–2014 inherit the noisier older elections' uncertainty. The middle bins contain few races, so the exact shape is noisy, but the direction is consistent. This is precisely the question the Bayesian models and Section 6 will address: whether treating older elections as a prior, rather than pooling them, gives better-calibrated forecasts.

## 5. Regression: by how much?

The regression models predict the two-party margin itself. Following the exploratory analysis (`01_explore.ipynb`, Section 2), they are trained and evaluated on **contested races only**: an uncontested race's margin of exactly ±1 says nothing about how competitive the district is. Section 5.2 tests what happens if those races are left in.

Each model is scored on **RMSE** (root mean squared error, in margin units, so 0.05 = 5 points), **MAE** (mean absolute error), and the **winner accuracy** implied by the sign of the predicted margin, overall and on competitive races.

In [ ]:
reg_df = df[df["contested"]]
reg_train, reg_val = years(reg_df, TRAIN_YEARS), years(reg_df, VAL_YEAR)
Xr_tr, yr_tr = Xy(reg_train, "margin")
Xr_val, yr_val = Xy(reg_val, "margin")

def reg_metrics(data, pred):
    y = data["margin"].to_numpy()
    comp = data["competitive"].to_numpy()
    won = data["dem_win"].to_numpy()
    called = (pred > 0).astype(int)
    return {"RMSE": np.sqrt(mean_squared_error(y, pred)),
            "MAE": mean_absolute_error(y, pred),
            "winner accuracy": accuracy_score(won, called),
            "winner accuracy (competitive)": accuracy_score(won[comp], called[comp])}

print(f"train: {len(reg_train):,} contested races   validation: {len(reg_val):,} contested races")

### 5.1 Least squares vs. least absolute deviations

- **Least squares** minimizes the squared residuals, which has a closed-form solution (the normal equations).
- **LAD** minimizes the absolute residuals (Lecture 4). It is less sensitive to large residuals, at the cost of no closed form. scikit-learn fits it as median (50th percentile) quantile regression, with `alpha=0` meaning no regularization.

Neither model has hyperparameters here, so the 2016 results are a direct comparison.

In [ ]:
ols = make_pipeline(StandardScaler(), LinearRegression()).fit(Xr_tr, yr_tr)
lad = make_pipeline(StandardScaler(), QuantileRegressor(quantile=0.5, alpha=0.0, solver="highs")).fit(Xr_tr, yr_tr)

reg_val_table = pd.DataFrame({
    "baseline (same margin as last time)": reg_metrics(reg_val, reg_val["margin_lag1"].to_numpy()),
    "least squares": reg_metrics(reg_val, ols.predict(Xr_val)),
    "LAD":           reg_metrics(reg_val, lad.predict(Xr_val)),
}).T
reg_val_table.to_csv(RESULTS / "validation_regression.csv")

reg_coefs = pd.DataFrame({"least squares": ols[-1].coef_, "LAD": lad[-1].coef_}, index=FEATURES)
display(reg_val_table)
reg_coefs

Both models cut the baseline's error substantially (RMSE from 0.25 to about 0.14). The baseline repeats last cycle's margin exactly, including ±1 for districts that were uncontested last time, while the models shrink extreme past margins toward the center and adjust for incumbency and the national environment.

Least squares and LAD are nearly identical on contested races, with LAD marginally better. That's expected: once uncontested races are removed there are few extreme residuals left for the two losses to treat differently. The coefficients tell the same story as the classifier: the previous margin and incumbency dominate, and the midterm-penalty term is negative.

The winner accuracy implied by the predicted margins matches the baseline. In 2016 the margin models mostly add information about *how close* a race will be, not *who* wins.

### 5.2 Outlier experiment: what if uncontested races are left in?

Lecture 4 showed that squared error lets a few large residuals pull the fitted line toward them, while absolute error resists. Uncontested races are a natural real-world version of this: their margins of exactly ±1 sit far from what the features predict. Here both models are retrained **with** the uncontested races included, then evaluated on the same contested 2016 races as before.

In [ ]:
all_train = years(df, TRAIN_YEARS)
Xa_tr, ya_tr = Xy(all_train, "margin")
print(f"Training races: {len(reg_train):,} contested only vs. {len(all_train):,} including uncontested "
      f"({1 - len(reg_train) / len(all_train):.1%} are uncontested)")

ols_all = make_pipeline(StandardScaler(), LinearRegression()).fit(Xa_tr, ya_tr)
lad_all = make_pipeline(StandardScaler(), QuantileRegressor(quantile=0.5, alpha=0.0, solver="highs")).fit(Xa_tr, ya_tr)

outlier_table = pd.DataFrame({
    ("least squares", "contested only"):     reg_metrics(reg_val, ols.predict(Xr_val)),
    ("least squares", "with uncontested"):   reg_metrics(reg_val, ols_all.predict(Xr_val)),
    ("LAD", "contested only"):               reg_metrics(reg_val, lad.predict(Xr_val)),
    ("LAD", "with uncontested"):             reg_metrics(reg_val, lad_all.predict(Xr_val)),
}).T
outlier_table.to_csv(RESULTS / "validation_outlier_experiment.csv")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, metric in zip(axes, ["RMSE", "MAE"]):
    vals = outlier_table[metric].unstack().reindex(["least squares", "LAD"])
    vals.plot.bar(ax=ax, rot=0, color=["steelblue", "indianred"])
    ax.set(title=f"2016 validation {metric}", xlabel="", ylabel=metric, ylim=(0, vals.values.max() * 1.35))
    ax.legend(title="training data", frameon=False, loc="upper left", ncol=2)
fig.suptitle("Effect of including uncontested races in training", y=1.02)
fig.tight_layout(); fig.savefig(FIGS / "outlier_experiment.png", dpi=150, bbox_inches="tight")
plt.show()

outlier_table

Adding the uncontested races hurts both models on contested 2016 races, but **least squares is hurt much more on MAE**: its MAE rises from 0.107 to 0.122, while LAD's rises only from 0.105 to 0.110. With squared error, the ±1 margins pull the fitted relationship toward them, the same effect as the outlier example in Lecture 4. LAD largely resists. Both models' RMSE rise by similar amounts, because RMSE itself squares the errors and is dominated by the races each model gets most wrong.

This supports the decision to train the regression models on contested races only. It also suggests LAD is the safer choice whenever extreme values can't be filtered out cleanly.

### 5.3 Bayesian linear regression with a historical prior

*To be completed after the Bayesian modeling lectures, following the formulation used in class.*

Plan: a prior on the regression weights estimated from the 1978–1998 elections, updated with 2000–2014. The posterior gives a **credible interval** for every predicted margin, and the evaluation will check whether those intervals are honest (about 90% of actual margins should fall inside the 90% intervals). Per the exploratory analysis (`01_explore.ipynb`, Section 3), the noise level should be estimated from recent elections, because results have become much more predictable since the 1980s.

## 6. Research question: historical prior vs. pooled data

*To be completed with the Bayesian models.*

Plan: compare three ways of using the training elections, each evaluated on 2016:

- **pooled:** all 1978–2014 elections weighted equally (the logistic and least-squares models above);
- **recent only:** 2000–2014, ignoring older elections;
- **historical prior:** 1978–1998 as a prior, updated with 2000–2014, at a range of prior strengths.

The comparison focuses on calibration (log loss, calibration error, interval coverage), since that is what the research question asks about.

## 7. Final evaluation on the test and stress-test elections

*Intentionally not run yet.* The 2018 test set should be used exactly once, after every model, including the Bayesian ones, has been finalized on 2016. This section will then report every model on 2018 and 2022 in one place: the metric tables, ROC curves, calibration plots, and interval coverage.

## 8. Discussion and conclusion

*To be written once the final results are in.*

Limitations to cover:
- **Redistricting:** after each census, `margin_lag1` describes a differently drawn district (the reason for the 2022 stress test). Mid-decade redraws in individual states are not flagged.
- **Incumbent matching:** incumbents are matched by first initial and last name, so an incumbent who moved to a new district after redistricting is counted as an open seat.
- **Excluded races:** third-party winners, and for regression, uncontested races.
- **Limited features:** the models use only past results and national context, with no candidate quality, fundraising, or polling.